# Explore Patient Timelines

Compare raw MEDS events with ETHOS tokenized output for individual patients.

**Setup:** Point `DATA_DIR` at a directory containing:
- `*.safetensors` shard(s)
- `vocab_t*.csv`
- `interval_estimates.json`
- `static_data.pickle`
- Optionally: `*.parquet` raw MEDS file(s)

In [19]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

DATA_DIR = Path.home() / "inspect-ethos-tokenization"
MEDS_DIR = DATA_DIR  # set to None if no raw MEDS parquet available

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import pickle
from datetime import datetime, timedelta, timezone

import altair as alt
import polars as pl
import torch as th

from ethos.constants import STATIC_DATA_FN
from ethos.constants import SpecialToken as ST
from ethos.datasets._sharded_data import ShardedData
from ethos.vocabulary import Vocabulary

alt.data_transformers.disable_max_rows()

# Load tokenized data
sharded = ShardedData(DATA_DIR)
vocab = Vocabulary.from_path(DATA_DIR)

with open(DATA_DIR / STATIC_DATA_FN, "rb") as f:
    static_data = pickle.load(f)

# Load raw MEDS if available
raw_meds = None
if MEDS_DIR is not None:
    parquet_files = list(Path(MEDS_DIR).glob("*.parquet"))
    if parquet_files:
        raw_meds = pl.concat([pl.read_parquet(f) for f in parquet_files])
        print(f"Loaded raw MEDS: {raw_meds.shape[0]} events, {raw_meds['subject_id'].n_unique()} patients")

print(f"Loaded {len(sharded.shards)} shard(s), vocab size: {len(vocab)}")

Loaded raw MEDS: 1931545 events, 986 patients
Loaded 1 shard(s), vocab size: 4523


## List Patients

In [21]:
def list_patients(sharded: ShardedData) -> pl.DataFrame:
    """Build a table of all patients with their token counts."""
    rows = []
    for shard_idx, shard in enumerate(sharded.shards):
        pids = shard["patient_ids"].tolist()
        offsets = shard["patient_offsets"].tolist()
        total = shard["tokens"].get_shape()[0]
        for i, pid in enumerate(pids):
            end = offsets[i + 1] if i + 1 < len(offsets) else total
            rows.append({"patient_id": pid, "n_tokens": end - offsets[i], "shard": shard_idx})
    return pl.DataFrame(rows).sort("patient_id")


def find_patient(sharded: ShardedData, patient_id: int) -> tuple[int, int]:
    """Return (global_start, global_end) token indices for a patient."""
    for shard in sharded.shards:
        pids = shard["patient_ids"].tolist()
        if patient_id in pids:
            idx = pids.index(patient_id)
            offsets = shard["patient_offsets"].tolist()
            total = shard["tokens"].get_shape()[0]
            start = shard["offset"] + offsets[idx]
            end = shard["offset"] + (offsets[idx + 1] if idx + 1 < len(offsets) else total)
            return start, end
    raise ValueError(f"Patient {patient_id} not found in any shard")


patients_df = list_patients(sharded)
print(f"{len(patients_df)} patients total")
patients_df.head(20)

882 patients total


patient_id,n_tokens,shard
i64,i64,i64
10005368,697,0
10008830,3702,0
10010993,251,0
10025133,89,0
10027690,72,0
…,…,…
10084571,435,0
10089171,2414,0
10116173,732,0


## Select a Patient

In [22]:
PATIENT_ID = patients_df["patient_id"][0]  # <-- change this to pick a patient

# Extract tokenized timeline
start, end = find_patient(sharded, PATIENT_ID)
token_ids = sharded.tokens[start:end].tolist()
times_us = sharded.times[start:end].tolist()
token_strs = vocab.decode(token_ids)

# Categorize tokens
SPECIAL_TOKENS = set(ST)
time_interval_tokens = set(vocab.time_interval_stokens)
quantile_tokens = set(vocab.quantile_stokens)


def get_category(token_str: str) -> str:
    if token_str in SPECIAL_TOKENS:
        return "special"
    if token_str in time_interval_tokens:
        return "time_interval"
    if token_str in quantile_tokens:
        return "quantile"
    return token_str.split("//")[0].lower() if "//" in token_str else "other"


timeline_df = pl.DataFrame({
    "position": list(range(len(token_ids))),
    "token_id": token_ids,
    "token": token_strs,
    "time_us": times_us,
    "category": [get_category(t) for t in token_strs],
}).with_columns(
    timestamp=pl.from_epoch(pl.col("time_us"), time_unit="us"),
)

# Show patient info
sd = static_data.get(PATIENT_ID, {})
print(f"Patient {PATIENT_ID}: {len(token_ids)} tokens")
for cat, val in sd.items():
    if cat == "MEDS_BIRTH" and val.get("time"):
        dob = datetime.fromtimestamp(val["time"][0] / 1e6, tz=timezone.utc)
        print(f"  DOB: {dob.date()}")
    else:
        codes = val.get("code", [])
        print(f"  {cat}: {codes[-1] if codes else 'unknown'}")

timeline_df.head(30)

Patient 10005368: 697 tokens
  BMI: BMI//Q1
  GENDER: GENDER//M
  MARITAL: MARITAL//SINGLE
  DOB: 2108-01-01
  RACE: RACE//WHITE


position,token_id,token,time_us,category,timestamp
i64,i64,str,i64,str,datetime[μs]
0,72,"""VITAL//BLOOD_PRESSURE""",5059238400000000,"""vital""",2130-04-28 00:00:00
1,31,"""Q7""",5059238400000000,"""quantile""",2130-04-28 00:00:00
2,32,"""Q8""",5059238400000000,"""quantile""",2130-04-28 00:00:00
3,214,"""30d-2mt""",5062003200000000,"""time_interval""",2130-05-30 00:00:00
4,131,"""ICD//PCS//0""",5062003200000000,"""icd""",2130-05-30 00:00:00
…,…,…,…,…,…
25,29,"""Q5""",5062112280000000,"""quantile""",2130-05-31 06:18:00
26,60,"""LAB//50902//MEQ/L""",5062112280000000,"""lab""",2130-05-31 06:18:00
27,25,"""Q2""",5062112280000000,"""quantile""",2130-05-31 06:18:00


## Tokenized Timeline

In [23]:
# Color by category
category_colors = {
    "special": "#e41a1c",
    "time_interval": "#999999",
    "quantile": "#ff7f00",
    "lab": "#377eb8",
    "vital": "#4daf4a",
    "icd": "#984ea3",
    "atc": "#a65628",
    "diagnosis": "#984ea3",
    "procedure": "#f781bf",
    "hospital_admission": "#e41a1c",
    "transfer": "#8dd3c7",
    "admission_type": "#bebada",
    "insurance": "#fdb462",
    "sofa": "#b3de69",
}

categories = timeline_df["category"].unique().sort().to_list()
colors = [category_colors.get(c, "#666666") for c in categories]

# Interactive timeline: tokens by position, colored by category
selection = alt.selection_interval(encodings=["x"])

base = alt.Chart(timeline_df.to_pandas()).encode(
    x=alt.X("timestamp:T", title="Time"),
    y=alt.Y("category:N", title=None, sort=categories),
    color=alt.Color("category:N", scale=alt.Scale(domain=categories, range=colors), legend=None),
    tooltip=["position:Q", "token:N", "category:N", "timestamp:T"],
)

overview = base.mark_circle(size=30, opacity=0.7).properties(
    width=800, height=200, title="Tokenized Timeline (select a region to zoom)"
).add_params(selection)

detail = base.mark_circle(size=60, opacity=0.8).encode(
    x=alt.X("timestamp:T", title="Time", scale=alt.Scale(domain=selection)),
).properties(width=800, height=200, title="Zoomed View")

overview & detail

alt.VConcatChart(...)

## Token-by-Token Strip View

Each token as a cell in sequence order — useful for seeing the exact token stream the model sees.

In [24]:
# Show a slice of the token stream as a strip chart
# Adjust START_POS and N_TOKENS to scroll through the timeline
START_POS = 0
N_TOKENS = 100

strip_df = timeline_df.filter(
    (pl.col("position") >= START_POS) & (pl.col("position") < START_POS + N_TOKENS)
).to_pandas()

strip = alt.Chart(strip_df).mark_rect().encode(
    x=alt.X("position:O", title="Token Position", axis=alt.Axis(labelAngle=-90, labelFontSize=8)),
    color=alt.Color("category:N", scale=alt.Scale(domain=categories, range=colors)),
    tooltip=["position:Q", "token:N", "category:N", "timestamp:T"],
).properties(width=max(N_TOKENS * 12, 400), height=40)

labels = alt.Chart(strip_df).mark_text(angle=270, fontSize=8, align="left", dx=0, dy=0).encode(
    x=alt.X("position:O"),
    text="token:N",
    y=alt.value(55),
).properties(width=max(N_TOKENS * 12, 400), height=200)

strip & labels

alt.VConcatChart(...)

## Raw MEDS vs Tokenized (Side-by-Side)

In [25]:
EXCLUDE_BIRTH = True  # Toggle to include/exclude birth events from MEDS chart

if raw_meds is not None:
    patient_meds = raw_meds.filter(pl.col("subject_id") == PATIENT_ID).sort("time")
    print(f"Raw MEDS: {len(patient_meds)} events for patient {PATIENT_ID}")

    # Derive category from MEDS code prefix
    meds_display = patient_meds.select(
        "subject_id", "time", "code", "numeric_value"
    ).with_columns(
        category=pl.col("code").str.split("//").list.first().str.to_lowercase(),
    ).filter(
        pl.col("time").is_not_null()
    )

    if EXCLUDE_BIRTH:
        meds_display = meds_display.filter(pl.col("code") != "MEDS_BIRTH")
        print("(birth events excluded — set EXCLUDE_BIRTH = False to include)")

    meds_display = meds_display.to_pandas()

    meds_categories = sorted(meds_display["category"].unique())

    # Build a unified color scale across both charts
    all_categories = sorted(set(meds_categories) | set(categories))
    all_colors = [category_colors.get(c, "#666666") for c in all_categories]
    unified_scale = alt.Scale(domain=all_categories, range=all_colors)

    selection2 = alt.selection_interval(encodings=["x"])

    # Raw MEDS chart
    meds_chart = alt.Chart(meds_display).mark_circle(size=30, opacity=0.7).encode(
        x=alt.X("time:T", title="Time"),
        y=alt.Y("category:N", title=None, sort=meds_categories),
        color=alt.Color("category:N", scale=unified_scale, legend=None),
        tooltip=["code:N", "numeric_value:Q", "category:N", "time:T"],
    ).properties(
        width=800, height=200, title="Raw MEDS Events (select to zoom)"
    ).add_params(selection2)

    # Tokenized chart (shared zoom)
    token_chart = alt.Chart(timeline_df.to_pandas()).mark_circle(size=30, opacity=0.7).encode(
        x=alt.X("timestamp:T", title="Time", scale=alt.Scale(domain=selection2)),
        y=alt.Y("category:N", title=None, sort=categories),
        color=alt.Color("category:N", scale=unified_scale, legend=None),
        tooltip=["position:Q", "token:N", "category:N", "timestamp:T"],
    ).properties(width=800, height=200, title="Tokenized Timeline")

    display(meds_chart & token_chart)
else:
    print("No raw MEDS data loaded — set MEDS_DIR to enable side-by-side comparison")

Raw MEDS: 413 events for patient 10005368
(birth events excluded — set EXCLUDE_BIRTH = False to include)


alt.VConcatChart(...)

## Time-Aligned Table: MEDS Events ↔ Tokens

Row-by-row comparison at each timestamp. This is the most useful view for spotting tokenization bugs:
missing events, wrong quantile bins, misplaced time intervals, etc.

In [26]:
if raw_meds is not None:
    # --- Build a per-timestamp token summary ---
    # Group tokens by timestamp, excluding time_interval tokens (they sit between events)
    event_tokens = timeline_df.filter(pl.col("category") != "time_interval")
    tokens_by_time = (
        event_tokens
        .group_by("timestamp")
        .agg(
            pl.col("token").alias("tokens"),
            pl.col("category").alias("token_categories"),
        )
        .sort("timestamp")
    )

    # --- Build a per-timestamp MEDS summary ---
    patient_meds_pl = raw_meds.filter(pl.col("subject_id") == PATIENT_ID).sort("time")
    meds_by_time = (
        patient_meds_pl
        .filter(pl.col("time").is_not_null())
        .with_columns(timestamp=pl.col("time"))
        .group_by("timestamp")
        .agg(
            pl.col("code").alias("meds_codes"),
            pl.col("numeric_value").alias("meds_values"),
        )
        .sort("timestamp")
    )

    # --- Join on timestamp ---
    aligned = meds_by_time.join(tokens_by_time, on="timestamp", how="full", coalesce=True).sort("timestamp")

    # Flag rows where one side is missing
    aligned = aligned.with_columns(
        status=pl.when(pl.col("meds_codes").is_null())
        .then(pl.lit("⚠ TOKEN ONLY"))
        .when(pl.col("tokens").is_null())
        .then(pl.lit("⚠ MEDS ONLY"))
        .otherwise(pl.lit("✓ both")),
    )

    # Display - show mismatches first, then everything
    mismatches = aligned.filter(pl.col("status") != "✓ both")
    print(f"=== {len(mismatches)} timestamps with a mismatch out of {len(aligned)} total ===\n")
    if len(mismatches) > 0:
        print("MISMATCHED TIMESTAMPS (one side missing):")
        print(mismatches.select("timestamp", "status", "meds_codes", "tokens"))

    print(f"\n=== Full aligned table ({len(aligned)} timestamps) ===")
    with pl.Config(
        tbl_cols=-1, tbl_rows=50, fmt_str_lengths=80, tbl_width_chars=200
    ):
        print(aligned.select("timestamp", "status", "meds_codes", "meds_values", "tokens", "token_categories"))
else:
    print("No raw MEDS data — skipping aligned table")

=== 57 timestamps with a mismatch out of 116 total ===

MISMATCHED TIMESTAMPS (one side missing):
shape: (57, 4)
┌─────────────────────┬─────────────┬─────────────────────────────────┬───────────┐
│ timestamp           ┆ status      ┆ meds_codes                      ┆ tokens    │
│ ---                 ┆ ---         ┆ ---                             ┆ ---       │
│ datetime[μs]        ┆ str         ┆ list[str]                       ┆ list[str] │
╞═════════════════════╪═════════════╪═════════════════════════════════╪═══════════╡
│ 2108-01-01 00:00:00 ┆ ⚠ MEDS ONLY ┆ ["MEDS_BIRTH"]                  ┆ null      │
│ 2130-05-30 15:00:00 ┆ ⚠ MEDS ONLY ┆ ["MEDICATION//START//Pantopraz… ┆ null      │
│ 2130-05-30 19:00:00 ┆ ⚠ MEDS ONLY ┆ ["MEDICATION//START//Ketorolac… ┆ null      │
│ 2130-05-30 20:00:00 ┆ ⚠ MEDS ONLY ┆ ["MEDICATION//START//Heparin"]  ┆ null      │
│ 2130-05-31 00:00:00 ┆ ⚠ MEDS ONLY ┆ ["MEDICATION//START//Lidocaine… ┆ null      │
│ …                   ┆ …           ┆ …        

## Detailed Event Diff

Expand a single timestamp to see every MEDS event and every token side by side.
Set `FOCUS_TIME` to a timestamp from the table above (or `None` to auto-pick the first mismatch).

In [27]:
FOCUS_TIME = None  # e.g. datetime(2130, 5, 30) — or None to auto-pick first mismatch

if raw_meds is not None:
    if FOCUS_TIME is None and len(mismatches) > 0:
        FOCUS_TIME = mismatches["timestamp"][0]
        print(f"Auto-selected first mismatch: {FOCUS_TIME}\n")
    elif FOCUS_TIME is None:
        FOCUS_TIME = aligned["timestamp"][0]
        print(f"No mismatches — showing first timestamp: {FOCUS_TIME}\n")

    # MEDS events at this time
    meds_at_t = patient_meds_pl.filter(
        pl.col("time") == FOCUS_TIME
    ).select("code", "numeric_value", "text_value")
    print(f"--- MEDS events at {FOCUS_TIME} ({len(meds_at_t)} events) ---")
    with pl.Config(tbl_rows=-1, fmt_str_lengths=120, tbl_width_chars=200):
        print(meds_at_t)

    # Tokens at this time
    tokens_at_t = timeline_df.filter(pl.col("timestamp") == FOCUS_TIME).select(
        "position", "token", "category"
    )
    print(f"\n--- Tokens at {FOCUS_TIME} ({len(tokens_at_t)} tokens) ---")
    with pl.Config(tbl_rows=-1, fmt_str_lengths=120, tbl_width_chars=200):
        print(tokens_at_t)
else:
    print("No raw MEDS data — skipping event diff")

Auto-selected first mismatch: 2108-01-01 00:00:00

--- MEDS events at 2108-01-01 00:00:00 (1 events) ---
shape: (1, 3)
┌────────────┬───────────────┬────────────┐
│ code       ┆ numeric_value ┆ text_value │
│ ---        ┆ ---           ┆ ---        │
│ str        ┆ f32           ┆ str        │
╞════════════╪═══════════════╪════════════╡
│ MEDS_BIRTH ┆ null          ┆ null       │
└────────────┴───────────────┴────────────┘

--- Tokens at 2108-01-01 00:00:00 (0 tokens) ---
shape: (0, 3)
┌──────────┬───────┬──────────┐
│ position ┆ token ┆ category │
│ ---      ┆ ---   ┆ ---      │
│ i64      ┆ str   ┆ str      │
╞══════════╪═══════╪══════════╡
└──────────┴───────┴──────────┘


## Quick Patient Cycler

Run `show_patient(patient_id)` to instantly load a new patient without re-running the whole notebook.
Use `show_patient(patient_id, time_range=(start, end))` to zoom into a date range.

In [28]:
def show_patient(pid: int, time_range: tuple[datetime, datetime] | None = None):
    """All-in-one debugger for a single patient. Prints aligned table and flags mismatches."""
    from IPython.display import display as ipy_display

    # --- tokens ---
    s, e = find_patient(sharded, pid)
    tids = sharded.tokens[s:e].tolist()
    tus = sharded.times[s:e].tolist()
    tstrs = vocab.decode(tids)

    tl = pl.DataFrame({
        "position": list(range(len(tids))),
        "token_id": tids,
        "token": tstrs,
        "time_us": tus,
        "category": [get_category(t) for t in tstrs],
    }).with_columns(timestamp=pl.from_epoch(pl.col("time_us"), time_unit="us"))

    # --- static data ---
    sd = static_data.get(pid, {})
    print(f"═══ Patient {pid}: {len(tids)} tokens ═══")
    for cat, val in sd.items():
        if cat == "MEDS_BIRTH" and val.get("time"):
            dob = datetime.fromtimestamp(val["time"][0] / 1e6, tz=timezone.utc)
            print(f"  DOB: {dob.date()}")
        else:
            codes = val.get("code", [])
            print(f"  {cat}: {codes[-1] if codes else 'unknown'}")

    # --- optional time filter ---
    if time_range is not None:
        t_start, t_end = time_range
        tl = tl.filter((pl.col("timestamp") >= t_start) & (pl.col("timestamp") <= t_end))
        print(f"  Filtered to {t_start} → {t_end}: {len(tl)} tokens")

    # --- aligned table (if MEDS available) ---
    if raw_meds is not None:
        pm = raw_meds.filter(pl.col("subject_id") == pid).sort("time")
        pm_ts = pm.filter(pl.col("time").is_not_null()).with_columns(
            timestamp=pl.col("time")
        )
        if time_range is not None:
            t_start, t_end = time_range
            pm_ts = pm_ts.filter((pl.col("timestamp") >= t_start) & (pl.col("timestamp") <= t_end))

        mbt = pm_ts.group_by("timestamp").agg(
            pl.col("code").alias("meds_codes"),
            pl.col("numeric_value").alias("meds_values"),
        ).sort("timestamp")

        evt = tl.filter(pl.col("category") != "time_interval")
        tbt = evt.group_by("timestamp").agg(
            pl.col("token").alias("tokens"),
        ).sort("timestamp")

        aln = mbt.join(tbt, on="timestamp", how="full", coalesce=True).sort("timestamp")
        aln = aln.with_columns(
            status=pl.when(pl.col("meds_codes").is_null()).then(pl.lit("⚠ TOKEN ONLY"))
            .when(pl.col("tokens").is_null()).then(pl.lit("⚠ MEDS ONLY"))
            .otherwise(pl.lit("✓")),
        )
        mm = aln.filter(pl.col("status") != "✓")
        print(f"\n  {len(mm)} mismatched timestamps / {len(aln)} total")
        if len(mm) > 0:
            print("\n  MISMATCHES:")
            with pl.Config(tbl_cols=-1, tbl_rows=30, fmt_str_lengths=80, tbl_width_chars=200):
                print(mm.select("timestamp", "status", "meds_codes", "tokens"))

        print(f"\n  Full aligned view:")
        with pl.Config(tbl_cols=-1, tbl_rows=50, fmt_str_lengths=80, tbl_width_chars=200):
            print(aln.select("timestamp", "status", "meds_codes", "meds_values", "tokens"))
    else:
        print("\n  (no MEDS data for comparison)")
        with pl.Config(tbl_rows=50, fmt_str_lengths=80):
            print(tl.select("position", "token", "category", "timestamp"))


# --- Example usage ---
# show_patient(10005368)
# show_patient(10005368, time_range=(datetime(2130, 5, 30), datetime(2130, 6, 1)))

## Batch Scan: Flag Patients with Potential Issues

Quickly scan all patients (or a sample) for common tokenization red flags.

In [ ]:
SCAN_LIMIT = 50  # set to None to scan all patients

_time_interval_tokens = set(vocab.time_interval_stokens)
_quantile_tokens = set(vocab.quantile_stokens)

scan_pids = patients_df["patient_id"].to_list()[:SCAN_LIMIT]
flagged = []

for pid in scan_pids:
    issues = []
    s, e = find_patient(sharded, pid)
    tids = sharded.tokens[s:e].tolist()
    tus = sharded.times[s:e].tolist()
    tstrs = vocab.decode(tids)
    n_tokens = len(tids)

    # Check 1: very short timelines
    if n_tokens < 5:
        issues.append(f"very short ({n_tokens} tokens)")

    # Check 2: timestamps not monotonically non-decreasing
    for i in range(1, len(tus)):
        if tus[i] < tus[i - 1]:
            issues.append(f"time goes backward at pos {i}: {tus[i-1]} -> {tus[i]}")
            break

    # Check 3: consecutive duplicate tokens (same token_id AND same time)
    for i in range(1, len(tids)):
        if tids[i] == tids[i - 1] and tus[i] == tus[i - 1]:
            cat = get_category(tstrs[i])
            if cat not in ("quantile", "time_interval"):
                issues.append(f"duplicate token at pos {i}: {tstrs[i]}")
                break

    # Check 4: quantile token not preceded by a code token
    for i in range(1, len(tstrs)):
        if tstrs[i].startswith("Q") and tstrs[i] in _quantile_tokens:
            prev_cat = get_category(tstrs[i - 1])
            if prev_cat == "time_interval" or prev_cat == "special":
                issues.append(f"orphan quantile at pos {i}: {tstrs[i]} after {tstrs[i-1]}")
                break

    # Check 5: missing from static_data
    if pid not in static_data:
        issues.append("missing from static_data.pickle")

    # Check 6: MEDS event count mismatch (if MEDS available)
    if raw_meds is not None:
        pm = raw_meds.filter(pl.col("subject_id") == pid)
        n_meds = len(pm)
        # Very rough: if token count is < 20% of MEDS events, something may be wrong
        non_time_tokens = sum(1 for t in tstrs if t not in _time_interval_tokens)
        if n_meds > 10 and non_time_tokens < n_meds * 0.1:
            issues.append(f"very few tokens ({non_time_tokens}) vs MEDS events ({n_meds})")

    if issues:
        flagged.append({"patient_id": pid, "n_tokens": n_tokens, "issues": "; ".join(issues)})

if flagged:
    flagged_df = pl.DataFrame(flagged)
    print(f"⚠ {len(flagged)} / {len(scan_pids)} patients flagged:\n")
    with pl.Config(tbl_rows=-1, fmt_str_lengths=120, tbl_width_chars=200):
        print(flagged_df)
    print("\nUse show_patient(<pid>) to investigate further.")
else:
    print(f"✓ No issues found in {len(scan_pids)} patients scanned.")

## Token Statistics

In [11]:
# Token type distribution
cat_counts = timeline_df.group_by("category").len().sort("len", descending=True)

bar = alt.Chart(cat_counts.to_pandas()).mark_bar().encode(
    x=alt.X("len:Q", title="Count"),
    y=alt.Y("category:N", title=None, sort="-x"),
    color=alt.Color("category:N", scale=alt.Scale(domain=categories, range=colors), legend=None),
    tooltip=["category:N", "len:Q"],
).properties(width=500, height=250, title="Token Type Distribution")

# Timeline duration
timeline_duration = timedelta(microseconds=int(times_us[-1] - times_us[0]))
if raw_meds is not None:
    _patient_meds = raw_meds.filter(pl.col("subject_id") == PATIENT_ID)
    n_raw = len(_patient_meds)
else:
    n_raw = None
print(f"Timeline duration: {timeline_duration}")
print(f"Token count: {len(token_ids)}")
print(f"Raw MEDS event count: {n_raw or 'N/A'}")
if n_raw is not None:
    print(f"Compression ratio: {n_raw / len(token_ids):.2f}x (raw events / tokens)")

bar